<a href="https://colab.research.google.com/github/hzhoujoy/ST554_HW6/blob/main/HW6_part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ST554_HomeWork_6   
Author: Huiping Zhou   
Date: 3/6/2026

# Part I-More Practice Querying a Database

We will continue working with the [lahman_1871-2022.sqlite](https://github.com/jknecht/baseball-archive-sqlite/releases/tag/2022) database, which contains information on Major League Baseball.

## Question 1
We will connect to the database and then look at all of the tables in the database by using `read_sql()` from pandas and returns it as a dataframe.

In [4]:
import sqlite3
import pandas as pd
# make the connection to the database
con = sqlite3.connect("lahman_1871-2022.sqlite")
#SQL query to return all table names in the data base
pd.read_sql("SELECT * FROM sqlite_master WHERE type='table';", con)

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


The lahman_1871-2022 database includes 27 tables.

## Question 2
Using SQL, construct a table of hall of fame pitchers (any hall of famer that pitched) that gives the playerID and their total (sum) for GS, G, W, L, IPOuts, CG, SHO, and SV columns.

First of all, let us check a few rows of table Pitching

In [5]:
pd.read_sql('SELECT * FROM Pitching LIMIT 5;',con)

,playerID,yearID,stint,teamID,lgID,W,L,G,GS,CG,...,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP
0,aardsda01,2004,1,SFN,NL,1,0,11,0,0,...,0,0,2,0,61,5,8,0,1,1
1,aardsda01,2006,1,CHN,NL,3,0,45,0,0,...,0,1,1,0,225,9,25,1,3,2
2,aardsda01,2007,1,CHA,AL,2,1,25,0,0,...,3,2,1,0,151,7,24,2,1,1
3,aardsda01,2008,1,BOS,AL,4,2,47,0,0,...,2,3,5,0,228,7,32,3,2,4
4,aardsda01,2009,1,SEA,AL,3,6,73,0,0,...,3,2,0,0,296,53,23,2,1,2


In [6]:
pd.read_sql('SELECT * FROM Batting LIMIT 5;',con) # check few rows of table Batting

,playerID,yearID,stint,teamID,lgID,G,G_batting,AB,R,H,...,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP,G_old
0,aardsda01,2004,1,SFN,NL,11,None,0,0,0,...,0,0,0,0,0,0,0,0,0,None
1,aardsda01,2006,1,CHN,NL,45,None,2,0,0,...,0,0,0,0,0,0,1,0,0,None
2,aardsda01,2007,1,CHA,AL,25,None,0,0,0,...,0,0,0,0,0,0,0,0,0,None
3,aardsda01,2008,1,BOS,AL,47,None,1,0,0,...,0,0,0,1,0,0,0,0,0,None
4,aardsda01,2009,1,SEA,AL,73,None,0,0,0,...,0,0,0,0,0,0,0,0,0,None


Then, we will construct a table named halloffame_pitchers by inner joining the Pitching and HallOfFame on playerID. We filter to only Hall of Fame inductees (inducted = 'Y') and return the column playerID, sum of GS, G, W, L, IPOuts, CG, SHO, and SV columns.

In [7]:
#create the cursor instance
cursor = con.cursor()
# create SQL string
ct='''
    CREATE TABLE halloffame_pitchers AS
    SELECT DISTINCT p.playerID,
    sum(p.GS) AS Total_GS,
    SUM(p.G) AS Total_G,
    SUM(p.W) AS Total_W,
    SUM(p.L) AS Total_L,
    SUM(p.IPOuts) AS Total_IPouts,
    SUM(p.CG) AS Total_CG,
    SUM(p.SHO) AS Total_SHO,
    SUM(p.SV) AS Total_SV
    FROM Pitching AS p
    INNER JOIN HallOfFame AS h ON p.playerID = h.playerID
    WHERE h.inducted = 'Y'
    Group BY p.playerID
    ORDER BY p.playerID;
 '''
 # Execute the CREATE TABLE statement
cursor.execute(ct)
# Commit the changes to the database
con.commit()
#close the cursor
cursor.close()
# Verify the table was created by querying it
pd.read_sql("SELECT * FROM halloffame_pitchers;", con)

,playerID,Total_GS,Total_G,Total_W,Total_L,Total_IPouts,Total_CG,Total_SHO,Total_SV
0,alexape01,599,696,373,208,15570,437,90,32
1,ansonca01,0,3,0,1,12,0,0,1
2,becklja01,1,1,0,1,12,0,0,0
3,bendech01,334,459,212,127,9051,255,40,34
4,blylebe01,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11
104,wrighge01,0,3,0,1,15,0,0,0
105,wrighha01,8,36,4,4,301,0,0,14
106,wynnea01,612,691,300,244,13692,290,49,15


## Question 3
To get the batting statistics specifically for Hall of Fame pitchers, we need to join the Batting, HallOfFame, and Pitching tables. This ensures that we only consider players who are inducted into the Hall of Fame and have pitching records. We will create a table named batting_stat for all of the hall of fame pitchers, including playerID, their total AB, R, H, HR, RBI, BB, and SO.

In [8]:
# create the cursor instance
cursor = con.cursor()
# create SQL string
ct_2 = '''
    CREATE TABLE batting_stat AS
    SELECT
        b.playerID,
        SUM(b.AB) AS Total_AB,
        SUM(b.R) AS Total_R,
        SUM(b.H) AS Total_H,
        SUM(b.HR) AS Total_HR,
        SUM(b.RBI) AS Total_RBI,
        SUM(b.BB) AS Total_BB,
        SUM(b.SO) AS Total_SO
    FROM Batting AS b
    INNER JOIN HallOfFame AS h ON b.playerID = h.playerID
    INNER JOIN Pitching AS p ON b.playerID = p.playerID
    WHERE h.inducted = 'Y'
    GROUP BY b.playerID
    ORDER BY b.playerID;
'''
# Execute the CREATE TABLE statement
cursor.execute(ct_2)
# Commit the changes to the database
con.commit()
#close the cursor
cursor.close()
# Verify the table was created by querying it
pd.read_sql("SELECT * FROM batting_stat;", con)

,playerID,Total_AB,Total_R,Total_H,Total_HR,Total_RBI,Total_BB,Total_SO
0,alexape01,38010,3234,7938,231,3423,1617,5796
1,ansonca01,20562,3998,6870,194,4150,1968,660
2,becklja01,9551,1603,2938,87,1581,616,526
3,bendech01,18352,1632,3888,96,1856,1200,2288
4,blylebe01,10824,456,1416,0,600,120,4632
...,...,...,...,...,...,...,...,...
103,willivi01,19409,1391,3224,13,1092,1053,2587
104,wrighge01,5746,1330,1732,22,652,136,238
105,wrighha01,3252,732,896,16,452,148,56
106,wynnea01,39192,3128,8395,391,3979,3243,7590


## Question 4
Using pandas join the previous two tables together by pitcher.

In [9]:
pd.merge(
 left = pd.read_sql("SELECT * FROM halloffame_pitchers", con),
 right = pd.read_sql("SELECT * FROM batting_stat", con),
 how = "left",
 on = "playerID")

,playerID,Total_GS,Total_G,Total_W,Total_L,Total_IPouts,Total_CG,Total_SHO,Total_SV,Total_AB,Total_R,Total_H,Total_HR,Total_RBI,Total_BB,Total_SO
0,alexape01,599,696,373,208,15570,437,90,32,38010,3234,7938,231,3423,1617,5796
1,ansonca01,0,3,0,1,12,0,0,1,20562,3998,6870,194,4150,1968,660
2,becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581,616,526
3,bendech01,334,459,212,127,9051,255,40,34,18352,1632,3888,96,1856,1200,2288
4,blylebe01,685,692,287,250,14910,242,60,0,10824,456,1416,0,600,120,4632
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,willivi01,471,513,249,205,11988,388,50,11,19409,1391,3224,13,1092,1053,2587
104,wrighge01,0,3,0,1,15,0,0,0,5746,1330,1732,22,652,136,238
105,wrighha01,8,36,4,4,301,0,0,14,3252,732,896,16,452,148,56
106,wynnea01,612,691,300,244,13692,290,49,15,39192,3128,8395,391,3979,3243,7590
